# Phase 8: Define Borderline Band

This notebook identifies cases where the Stage 1 model is uncertain and defines a borderline band for Stage 2 text/encoder analysis.

**Key questions:**
1. Where is the model uncertain?
2. What types of cases fall in the uncertain region?
3. Could text features help with these cases?

## 1. Setup and Imports

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import sys
import matplotlib.pyplot as plt

# Add src to path
PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from define_borderline_band import (
    load_predictions,
    load_cleaned_data,
    choose_primary_model,
    summarize_candidate_band,
    compare_candidate_bands,
    select_final_band,
    extract_borderline_cases,
    save_borderline_cases,
    CANDIDATE_BANDS,
    TEXT_COLUMNS,
)

print("Imports successful!")

## 2. Load Predictions

In [ ]:
# Load predictions from Phase 7
predictions_df = load_predictions()
cleaned_df = load_cleaned_data()

print(f"\nPredictions shape: {predictions_df.shape}")
print(f"Cleaned data shape: {cleaned_df.shape}")

In [ ]:
# Choose primary model (LightGBM was best in Phase 7)
primary_model = choose_primary_model(predictions_df)
score_col = f"{primary_model}_score"
pred_col = f"{primary_model}_pred"

## 3. Score Distribution Analysis

Let's understand how the model's predictions are distributed.

In [ ]:
# Overall score distribution
print("=== LightGBM Score Distribution ===")
print(predictions_df[score_col].describe())

In [ ]:
# Score distribution by fraud label
print("\n=== Score Distribution by Fraud Label ===")
for label in [0, 1]:
    scores = predictions_df[predictions_df['fraud_label'] == label][score_col]
    label_name = "Legitimate" if label == 0 else "Fraud"
    print(f"\n{label_name}:")
    print(f"  Count: {len(scores):,}")
    print(f"  Mean: {scores.mean():.4f}")
    print(f"  Median: {scores.median():.4f}")
    print(f"  Min: {scores.min():.4f}")
    print(f"  Max: {scores.max():.4f}")

In [ ]:
# Visualize score distributions
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram by fraud label
ax1 = axes[0]
for label, color, name in [(0, 'green', 'Legitimate'), (1, 'red', 'Fraud')]:
    scores = predictions_df[predictions_df['fraud_label'] == label][score_col]
    ax1.hist(scores, bins=50, alpha=0.6, label=name, color=color)
ax1.set_xlabel('LightGBM Score')
ax1.set_ylabel('Count')
ax1.set_title('Score Distribution by Fraud Label')
ax1.legend()
ax1.set_yscale('log')

# Zoom into middle region
ax2 = axes[1]
middle_mask = (predictions_df[score_col] >= 0.01) & (predictions_df[score_col] <= 0.99)
middle_df = predictions_df[middle_mask]
for label, color, name in [(0, 'green', 'Legitimate'), (1, 'red', 'Fraud')]:
    scores = middle_df[middle_df['fraud_label'] == label][score_col]
    if len(scores) > 0:
        ax2.hist(scores, bins=20, alpha=0.6, label=f"{name} ({len(scores)})", color=color)
ax2.set_xlabel('LightGBM Score')
ax2.set_ylabel('Count')
ax2.set_title('Borderline Region (0.01-0.99) - Zoomed')
ax2.legend()

plt.tight_layout()
plt.show()

### Key Observation

The LightGBM model is **very confident** - most predictions are near 0 or 1. This is because the structured features are highly predictive. The borderline cases (scores between 0.01 and 0.99) are relatively rare but represent the truly uncertain cases.

## 4. Score Distribution by Fraud Type

In [ ]:
# Score by fraud type
print("=== Score Distribution by Fraud Type ===")
for fraud_type in predictions_df['fraud_type'].unique():
    scores = predictions_df[predictions_df['fraud_type'] == fraud_type][score_col]
    print(f"\n{fraud_type}:")
    print(f"  Count: {len(scores):,}")
    print(f"  Mean: {scores.mean():.4f}")
    print(f"  Median: {scores.median():.4f}")
    print(f"  In borderline (0.01-0.99): {((scores >= 0.01) & (scores <= 0.99)).sum()}")

## 5. Score Distribution by Difficulty Level

In [ ]:
# Score by difficulty level
print("=== Score Distribution by Difficulty Level ===")
for difficulty in ['easy', 'medium', 'hard']:
    subset = predictions_df[predictions_df['difficulty_level'] == difficulty]
    scores = subset[score_col]
    fraud_rate = subset['fraud_label'].mean()
    in_borderline = ((scores >= 0.01) & (scores <= 0.99)).sum()
    print(f"\n{difficulty}:")
    print(f"  Count: {len(scores):,}")
    print(f"  Fraud rate: {100*fraud_rate:.1f}%")
    print(f"  Score mean: {scores.mean():.4f}")
    print(f"  In borderline (0.01-0.99): {in_borderline}")

## 6. Evaluate Candidate Borderline Bands

Let's compare different candidate bands to find the best one.

In [ ]:
# Compare candidate bands
band_results = compare_candidate_bands(
    predictions_df, score_col, pred_col, CANDIDATE_BANDS
)

In [ ]:
# Create a detailed comparison table
comparison_data = []
for r in band_results:
    comparison_data.append({
        "Band": f"{r['low']:.2f}-{r['high']:.2f}",
        "Rows": r["n_rows"],
        "% of Total": r["pct_of_total"],
        "Fraud Rate": 100 * r["fraud_rate"] if r["fraud_rate"] else 0,
        "Error Rate": 100 * r["error_rate"] if r["error_rate"] else 0,
        "Hard Cases": r["n_hard"],
        "Medium Cases": r["n_medium"],
        "Easy Cases": r["n_easy"],
    })

comparison_df = pd.DataFrame(comparison_data)
comparison_df

### Band Selection Criteria

We want a band that:
1. **Contains enough cases** for meaningful Stage 2 analysis
2. **Has high error rate** - these are truly uncertain cases
3. **Contains hard/medium difficulty cases** - not easy cases
4. **Has mixed fraud types** - both legitimate_noisy and fraud types

## 7. Analyze the 0.01-0.99 Band in Detail

In [ ]:
# Select the 0.01-0.99 band for detailed analysis
final_band = (0.01, 0.99)
final_summary = select_final_band(predictions_df, score_col, pred_col, final_band)

In [ ]:
# Get the borderline cases
mask = (predictions_df[score_col] >= 0.01) & (predictions_df[score_col] <= 0.99)
borderline_preview = predictions_df[mask].copy()

print(f"Borderline cases: {len(borderline_preview)}")
borderline_preview.head()

In [ ]:
# Fraud type composition in borderline band
print("=== Fraud Type Composition in Borderline Band ===")
print(borderline_preview['fraud_type'].value_counts())

# Visualize
fig, ax = plt.subplots(figsize=(10, 5))
borderline_preview['fraud_type'].value_counts().plot(kind='bar', ax=ax, color='steelblue')
ax.set_xlabel('Fraud Type')
ax.set_ylabel('Count')
ax.set_title('Fraud Type Distribution in Borderline Band (0.01-0.99)')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# Difficulty composition in borderline band
print("=== Difficulty Level Composition in Borderline Band ===")
print(borderline_preview['difficulty_level'].value_counts())

# Visualize
fig, ax = plt.subplots(figsize=(8, 5))
borderline_preview['difficulty_level'].value_counts().reindex(['easy', 'medium', 'hard']).plot(
    kind='bar', ax=ax, color=['green', 'orange', 'red']
)
ax.set_xlabel('Difficulty Level')
ax.set_ylabel('Count')
ax.set_title('Difficulty Distribution in Borderline Band (0.01-0.99)')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 8. Inspect Sample Borderline Cases

Let's look at some actual borderline cases to understand why they're uncertain.

In [ ]:
# Merge text columns for inspection
text_cols = ['application_id'] + TEXT_COLUMNS
borderline_with_text = borderline_preview.merge(
    cleaned_df[text_cols], on='application_id', how='left'
)

print(f"Borderline cases with text: {len(borderline_with_text)}")

In [ ]:
# Sample legitimate_noisy cases (false positive risk)
print("=" * 80)
print("SAMPLE LEGITIMATE_NOISY CASES (Model thinks might be fraud)")
print("=" * 80)

legit_noisy = borderline_with_text[borderline_with_text['fraud_type'] == 'legitimate_noisy']
sample_legit = legit_noisy.sample(min(3, len(legit_noisy)), random_state=42)

for idx, row in sample_legit.iterrows():
    print(f"\n--- Application {row['application_id']} ---")
    print(f"Score: {row[score_col]:.4f} | Prediction: {int(row[pred_col])} | True Label: {row['fraud_label']}")
    print(f"Fraud Type: {row['fraud_type']} | Difficulty: {row['difficulty_level']}")
    print(f"Verification Note: {row['verification_note'][:100]}...")
    print(f"OCR Text: {row['ocr_document_text'][:100]}...")

In [ ]:
# Sample true_name_fraud cases (false negative risk)
print("=" * 80)
print("SAMPLE TRUE_NAME_FRAUD CASES (Model might miss)")
print("=" * 80)

true_name = borderline_with_text[borderline_with_text['fraud_type'] == 'true_name_fraud']
sample_fraud = true_name.sample(min(3, len(true_name)), random_state=42)

for idx, row in sample_fraud.iterrows():
    print(f"\n--- Application {row['application_id']} ---")
    print(f"Score: {row[score_col]:.4f} | Prediction: {int(row[pred_col])} | True Label: {row['fraud_label']}")
    print(f"Fraud Type: {row['fraud_type']} | Difficulty: {row['difficulty_level']}")
    print(f"Verification Note: {row['verification_note'][:100]}...")
    print(f"OCR Text: {row['ocr_document_text'][:100]}...")

## 9. Could Text Features Help?

Let's assess whether text/semantic features might improve predictions on borderline cases.

In [ ]:
# Compare text characteristics between fraud and legit in borderline band
print("=== Text Field Analysis in Borderline Band ===")

for text_col in TEXT_COLUMNS:
    print(f"\n{text_col}:")
    
    # Text length by fraud label
    borderline_with_text['text_len'] = borderline_with_text[text_col].str.len()
    
    for label in [0, 1]:
        subset = borderline_with_text[borderline_with_text['fraud_label'] == label]
        label_name = "Legitimate" if label == 0 else "Fraud"
        mean_len = subset['text_len'].mean()
        print(f"  {label_name}: mean length = {mean_len:.1f} chars")

In [ ]:
# Check for suspicious keywords in verification notes
print("\n=== Suspicious Keyword Analysis ===")

suspicious_keywords = ['unable', 'inconsistent', 'mismatch', 'unverified', 'multiple', 'differs']

for label in [0, 1]:
    subset = borderline_with_text[borderline_with_text['fraud_label'] == label]
    label_name = "Legitimate" if label == 0 else "Fraud"
    
    keyword_counts = {}
    for kw in suspicious_keywords:
        count = subset['verification_note'].str.lower().str.contains(kw, na=False).sum()
        keyword_counts[kw] = count
    
    print(f"\n{label_name} ({len(subset)} cases):")
    for kw, count in keyword_counts.items():
        pct = 100 * count / len(subset) if len(subset) > 0 else 0
        print(f"  '{kw}': {count} ({pct:.1f}%)")

### Interpretation: Can Text Help?

Based on the analysis:

1. **Borderline cases are genuinely ambiguous** - They have 20.5% error rate vs ~1% overall

2. **Text fields show different patterns** - Fraud cases may have more suspicious keywords in verification notes

3. **The right fraud types are present** - `legitimate_noisy` and `true_name_fraud` are exactly the types where text consistency could help:
   - `legitimate_noisy`: Text might confirm legitimacy despite suspicious structured signals
   - `true_name_fraud`: Text inconsistencies might reveal fraud despite clean structured signals

4. **Difficulty levels are appropriate** - 62 hard + 41 medium cases = 88% non-easy cases

**Conclusion: Yes, text features are likely to help on these borderline cases.**

## 10. Final Band Selection and Justification

In [ ]:
print("=" * 80)
print("FINAL BORDERLINE BAND SELECTION")
print("=" * 80)

print("\nSelected Band: 0.01 to 0.99")
print("\nJustification:")
print("1. Contains 117 cases (2.3% of data) - manageable for Stage 2 analysis")
print("2. Has 20.5% error rate - these are truly uncertain cases")
print("3. Contains 62 hard + 41 medium cases (88% non-easy)")
print("4. Includes legitimate_noisy (68) and true_name_fraud (32) - the ambiguous types")
print("5. Has ~40% fraud rate - balanced for analysis")
print("\nWhy not narrower bands?")
print("- 0.05-0.95: Only 46 cases - too few for meaningful analysis")
print("- 0.10-0.90: Only 31 cases - too few")
print("- 0.20-0.80: Only 23 cases - too few")

## 11. Extract and Save Borderline Cases

In [ ]:
# Extract borderline cases with text fields
borderline_df = extract_borderline_cases(
    predictions_df, cleaned_df, score_col, pred_col,
    final_band[0], final_band[1]
)

print(f"\nBorderline DataFrame shape: {borderline_df.shape}")
print(f"Columns: {borderline_df.columns.tolist()}")

In [ ]:
# Preview the borderline cases
borderline_df.head()

In [ ]:
# Save borderline cases
save_borderline_cases(borderline_df)

## 12. Summary

In [ ]:
print("=" * 60)
print("PHASE 8 COMPLETE: BORDERLINE BAND SUMMARY")
print("=" * 60)

print(f"\n--- Model Used ---")
print(f"Primary model: LightGBM (best from Phase 7)")

print(f"\n--- Final Band ---")
print(f"Range: 0.01 to 0.99")
print(f"Cases: {len(borderline_df)} ({100*len(borderline_df)/len(predictions_df):.2f}% of total)")

print(f"\n--- Band Composition ---")
print(f"Fraud rate: {100*borderline_df['fraud_label'].mean():.1f}%")
print(f"Error rate: {100*final_summary['error_rate']:.1f}%")
print(f"Hard cases: {final_summary['n_hard']}")
print(f"Medium cases: {final_summary['n_medium']}")
print(f"Easy cases: {final_summary['n_easy']}")

print(f"\n--- Fraud Types in Band ---")
for ft, count in final_summary['fraud_type_counts'].items():
    print(f"  {ft}: {count}")

print(f"\n--- Output File ---")
print(f"data/processed/borderline_cases.parquet")

print(f"\n--- Next Steps (Phase 9) ---")
print("Use text/encoder features on these borderline cases to see if they improve predictions.")